# Session 1: heat in, work out, heat rejected
**A. Real-world problem.** A car engine burns fuel, but only part of the fuel's energy
turns the wheels. The rest leaves through the exhaust and the radiator. Why can no engine
turn all the heat into work? And how can a loop drawn on a pressure-volume graph tell you
how much work one cycle gives?

**Plan.** (1) the energy balance W = Qh - Qc, (2) efficiency = W / Qh, (3) the Carnot
limit that no engine can beat, (4) the area of a P-V loop is the work.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the energy balance and the efficiency
Over one full cycle the engine returns to where it started, so its stored energy has not
changed. Energy in must equal energy out:

    Qh = W + Qc      so      W = Qh - Qc        efficiency = W / Qh

Qh is heat taken from the hot source, Qc is heat rejected to the cold surroundings, W is
the useful work. All three are in joules (J) per cycle.

In [ ]:
def work(q_hot, q_cold):
    """Net work per cycle (J) from the energy balance W = Qh - Qc."""
    return q_hot - q_cold


def efficiency(q_hot, q_cold):
    """Thermal efficiency W / Qh (no unit). Qh must be positive."""
    if q_hot <= 0:
        raise ValueError("heat taken from the hot source must be positive")
    return work(q_hot, q_cold) / q_hot

## Step 2: the Carnot limit
The second law of thermodynamics says some heat must always be rejected. If the hot source
is at temperature Th and the cold surroundings at Tc (both in **kelvin**, K = degrees C +
273.15), the best any engine can do is

    efficiency <= 1 - Tc / Th

In [ ]:
def carnot_limit(t_hot, t_cold):
    """Highest possible efficiency between two temperatures in kelvin."""
    if t_cold <= 0 or t_hot < t_cold:
        raise ValueError("need 0 < Tc <= Th, both in kelvin")
    return 1.0 - t_cold / t_hot


def is_allowed(q_hot, q_cold, t_hot, t_cold):
    """True if an engine with these heats does not break the energy balance or the Carnot limit."""
    if q_cold < 0 or q_cold > q_hot:
        return False
    return efficiency(q_hot, q_cold) <= carnot_limit(t_hot, t_cold) + 1e-12

## Step 3: a P-V loop and its area
Plot the gas pressure P against its volume V while the engine goes round one cycle. When
the gas expands at high pressure it does more work than is spent squeezing it back at low
pressure, so a loop that goes **clockwise** gives net work. The enclosed area is that work.
With pressure in kPa and volume in litres, kPa x L = J.

In [ ]:
def rectangle_loop(p_low, p_high, v_low, v_high):
    """Corners of a clockwise rectangular loop as (V, P) arrays: up, right, down, left."""
    v = np.array([v_low, v_low, v_high, v_high])
    p = np.array([p_low, p_high, p_high, p_low])
    return v, p


def loop_work(v, p):
    """Work done by the gas around a closed loop: the sum of P dV (trapezoid rule).

    Positive for a clockwise loop (an engine), negative for a counter-clockwise loop
    (a refrigerator or heat pump)."""
    v = np.asarray(v, float)
    p = np.asarray(p, float)
    v2, p2 = np.roll(v, -1), np.roll(p, -1)
    return float(np.sum(0.5 * (p + p2) * (v2 - v)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)

    # Figure 1: where does the heat go? three engines taking 1000 J from a 600 K source
    q_hot, t_hot, t_cold = 1000.0, 600.0, 300.0
    names = ["Engine A", "Engine B", "Engine C"]
    q_cold = [700.0, 550.0, 400.0]
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for i, (n, qc) in enumerate(zip(names, q_cold)):
        w = work(q_hot, qc)
        ax.bar(i, w, color="#2a9d8f", label="work W" if i == 0 else None)
        ax.bar(i, qc, bottom=w, color="#d9534f", alpha=0.6, label="rejected heat Qc" if i == 0 else None)
        ax.text(i, q_hot + 15, f"efficiency {efficiency(q_hot, qc):.2f}", ha="center", fontsize=9)
        print(f"{n}: W = {w:.0f} J, efficiency = {efficiency(q_hot, qc):.2f}, allowed = {is_allowed(q_hot, qc, t_hot, t_cold)}")
    ax.axhline(q_hot, color="k", lw=1, ls="--")
    ax.text(2.45, q_hot - 60, "Qh = 1000 J", ha="right", fontsize=9)
    ax.set_xticks(range(3))
    ax.set_xticklabels(names)
    ax.set_ylabel("heat per cycle (J)")
    ax.set_ylim(0, 1150)
    ax.set_title(f"Qh = W + Qc (Carnot limit {carnot_limit(t_hot, t_cold):.2f} for 600 K and 300 K)")
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "energy_split.png", dpi=120)

    # Figure 2: the area of a P-V loop is the work
    fig2, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
    for ax, (p_hi, name) in zip(axes, [(300.0, "small loop"), (500.0, "taller loop")]):
        v, p = rectangle_loop(100.0, p_hi, 1.0, 3.0)
        ax.fill(v, p, color="#2a9d8f", alpha=0.35)
        ax.plot(np.r_[v, v[0]], np.r_[p, p[0]], "k-")
        for k, lab in enumerate("1234"):
            ax.annotate(lab, (v[k], p[k]), textcoords="offset points", xytext=(6, 6))
        ax.annotate("", xy=(2.0, p_hi), xytext=(1.5, p_hi), arrowprops=dict(arrowstyle="->"))
        ax.set_title(f"{name}: W = {loop_work(v, p):.0f} J")
        ax.set_xlabel("volume V (L)")
        ax.grid(alpha=0.3)
        print(f"{name}: W = {loop_work(v, p):.0f} J")
    axes[0].set_ylabel("pressure P (kPa)")
    fig2.tight_layout()
    fig2.savefig(OUT / "pv_loop_area.png", dpi=120)

## Where this stops
A rectangle is not what a real engine draws. Session 2 builds the real shapes (the Otto and
Diesel cycles) from the gas laws and gets their efficiency from a formula.

## Coding tasks
1. **Easy.** In `lesson.py` change `q_cold` to 500, 450 and 350 J. Predict which engines are allowed between 600 K and 300 K, then run it.
2. **Stretch.** Write `refrigerator_loop` that returns the same rectangle traversed counter-clockwise and check that its work is minus 400 J.
3. **Explain.** Explain why the Carnot limit cannot be reached in practice, and what Tc and Th you would need for an efficiency of 0.9.

## Assessment
- **A. Conceptual.** Why can an engine never turn all of the heat it receives into work, even if it had no friction at all?
- **B. Numerical.** An engine takes 800 J from a hot source and rejects 520 J. Find the work and the efficiency, and check the units and the answer is reasonable.
- **C. Graph and simulation interpretation.** In the P-V view the loop is the same width but taller after you raise the work. By what factor does the area grow when the top pressure goes from 300 kPa to 500 kPa (bottom 100 kPa, volume 1 to 3 L)? What are the two works?
- **D. Design and optimisation challenge.** Design: an engine must give 300 J of work per cycle between a 700 K source and 300 K surroundings. What is the smallest heat Qh it must take in, and the smallest Qc? A real engine reaches 0.30. How much heat would it need?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: Qh = 1000 J, Qc = 650 J gives W = 350 J and efficiency 0.35",
                   abs(L.work(1000, 650) - 350) < 1e-9 and abs(L.efficiency(1000, 650) - 0.35) < 1e-12))
    v, p = L.rectangle_loop(100.0, 300.0, 1.0, 3.0)
    checks.append(("worked example: rectangle 100 to 300 kPa and 1 to 3 L encloses 400 J", abs(L.loop_work(v, p) - 400.0) < 1e-9))
    checks.append(("hand calculation: area = (300 - 100) kPa x (3 - 1) L", abs(L.loop_work(v, p) - (300 - 100) * (3 - 1)) < 1e-9))
    checks.append(("a counter-clockwise loop gives negative work", abs(L.loop_work(v[::-1], p[::-1]) + 400.0) < 1e-9))
    checks.append(("taller loop (100 to 500 kPa) gives 800 J", abs(L.loop_work(*L.rectangle_loop(100, 500, 1, 3)) - 800.0) < 1e-9))
    checks.append(("a loop that starts at another corner has the same area",
                   abs(L.loop_work(np.roll(v, 1), np.roll(p, 1)) - 400.0) < 1e-9))
    checks.append(("Carnot limit between 600 K and 300 K is 0.5", abs(L.carnot_limit(600, 300) - 0.5) < 1e-12))
    checks.append(("zero efficiency when Qc = Qh", L.efficiency(500, 500) == 0.0))
    checks.append(("engine with efficiency 0.6 between 600 K and 300 K is not allowed", not L.is_allowed(1000, 400, 600, 300)))
    checks.append(("engine with efficiency 0.5 exactly at the limit is allowed (round-off safe)", L.is_allowed(1000, 500, 600, 300)))
    checks.append(("Qc larger than Qh is not allowed", not L.is_allowed(1000, 1200, 600, 300)))
    bad = 0
    for call in (lambda: L.efficiency(0, 0), lambda: L.carnot_limit(300, 600), lambda: L.carnot_limit(600, 0)):
        try:
            call()
        except ValueError:
            bad += 1
    checks.append(("invalid inputs raise ValueError", bad == 3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
